# Model Training and Benchmarking (Train/Test Only)
Benchmark multiple classifiers with Stratified K-fold cross-validation, select the best model by ROC-AUC, fit on full train data, then generate test predictions.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

train_df = pd.read_csv('../data/train_fe.csv')
test_df = pd.read_csv('../data/test_fe.csv')

selected_features_path = Path('../models/selected_features_gsa.json')
selected_features = None
if selected_features_path.exists():
    with selected_features_path.open('r', encoding='utf-8') as f:
        payload = json.load(f)
    selected_features = payload.get('selected_features_raw') or payload.get('selected_features') or []
    print('Loaded selected raw features from EDA:', len(selected_features))
else:
    print('No selected feature file found. Use all engineered features.')

# Optional boosting libraries
xgb_available = False
cat_available = False
try:
    from xgboost import XGBClassifier
    xgb_available = True
except Exception:
    print('xgboost not installed: skip XGBoost model')

try:
    from catboost import CatBoostClassifier
    cat_available = True
except Exception:
    print('catboost not installed: skip CatBoost model')

In [ ]:
y = train_df['Churn'].astype(str).str.strip().str.lower().map({'yes': 1, 'no': 0})
valid_idx = y.notna()
if valid_idx.mean() < 1.0:
    print('Dropped rows with invalid target labels:', int((~valid_idx).sum()))

X_full = train_df.loc[valid_idx].drop(columns=['Churn', 'id'])
y = y.loc[valid_idx].astype(int)
X_test_full = test_df.drop(columns=['id'])

if selected_features:
    selected_features = [c for c in selected_features if c in X_full.columns]
    if len(selected_features) > 0:
        X = X_full[selected_features].copy()
        X_test = X_test_full.reindex(columns=selected_features).copy()
        print('Training with selected features:', len(selected_features))
    else:
        X = X_full.copy()
        X_test = X_test_full.copy()
        print('Selected features not found in train_fe. Fallback to all features.')
else:
    X = X_full.copy()
    X_test = X_test_full.copy()

num_cols = X.select_dtypes(include=['number', 'bool']).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', ohe)]), cat_cols),
])

In [ ]:
def build_preprocessor(scale_numeric: bool):
    num_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        num_steps.append(('scaler', StandardScaler()))

    try:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

    return ColumnTransformer([
        ('num', Pipeline(num_steps), num_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', ohe)]), cat_cols),
    ])

model_specs = [
    ('LogisticRegression', LogisticRegression(class_weight='balanced', random_state=42, max_iter=3000), True),
    ('DecisionTree', DecisionTreeClassifier(random_state=42, max_depth=6, min_samples_leaf=20), False),
    ('RandomForest', RandomForestClassifier(random_state=42, n_estimators=400, max_depth=10, min_samples_leaf=5, n_jobs=-1), False),
    ('GaussianNB', GaussianNB(), True),
    ('SVM_Linear', SVC(kernel='linear', C=1.0, probability=True, class_weight='balanced', random_state=42), True),
    ('SVM_Poly', SVC(kernel='poly', degree=3, C=1.0, probability=True, class_weight='balanced', random_state=42), True),
    ('KNN', KNeighborsClassifier(n_neighbors=15, weights='distance'), True),
    ('AdaBoost', AdaBoostClassifier(random_state=42, n_estimators=300, learning_rate=0.05), False),
]

if xgb_available:
    model_specs.append((
        'XGBoost',
        XGBClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.9,
            colsample_bytree=0.9,
            objective='binary:logistic',
            eval_metric='logloss',
            random_state=42,
            n_jobs=-1,
        ),
        False
    ))

if cat_available:
    model_specs.append((
        'CatBoost',
        CatBoostClassifier(
            loss_function='Logloss',
            depth=6,
            learning_rate=0.05,
            n_estimators=500,
            random_seed=42,
            verbose=False,
        ),
        False
    ))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
best_name = None
best_pipeline = None
best_cv_auc = -np.inf

for name, estimator, use_scaler in model_specs:
    preprocessor_i = build_preprocessor(scale_numeric=use_scaler)
    pipeline_i = Pipeline([('preprocessor', preprocessor_i), ('model', estimator)])

    auc_scores = cross_val_score(pipeline_i, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    ap_scores = cross_val_score(pipeline_i, X, y, cv=cv, scoring='average_precision', n_jobs=-1)

    auc_mean = float(np.mean(auc_scores))
    row = {
        'model': name,
        'cv_roc_auc_mean': auc_mean,
        'cv_roc_auc_std': float(np.std(auc_scores)),
        'cv_ap_mean': float(np.mean(ap_scores)),
        'cv_ap_std': float(np.std(ap_scores)),
    }
    rows.append(row)

    if auc_mean > best_cv_auc:
        best_cv_auc = auc_mean
        best_name = name
        best_pipeline = pipeline_i

benchmark_df = pd.DataFrame(rows).sort_values('cv_roc_auc_mean', ascending=False).reset_index(drop=True)
display(benchmark_df)
print('Best model from K-fold CV:', best_name, '| ROC-AUC:', round(best_cv_auc, 5))

pipeline = best_pipeline
pipeline.fit(X, y)
train_proba = pipeline.predict_proba(X)[:, 1]

metrics = {
    'selected_model': best_name,
    'train_roc_auc': roc_auc_score(y, train_proba),
    'train_average_precision': average_precision_score(y, train_proba),
    'cv_roc_auc_mean': float(benchmark_df.loc[0, 'cv_roc_auc_mean']),
    'cv_roc_auc_std': float(benchmark_df.loc[0, 'cv_roc_auc_std']),
    'cv_ap_mean': float(benchmark_df.loc[0, 'cv_ap_mean']),
    'cv_ap_std': float(benchmark_df.loc[0, 'cv_ap_std']),
}
metrics

In [ ]:
test_proba = pipeline.predict_proba(X_test)[:, 1]
pd.Series(test_proba).describe()

In [ ]:
import json
import joblib

joblib.dump(pipeline, '../models/churn_model_train_test_only.joblib')
print('Saved ../models/churn_model_train_test_only.joblib')

benchmark_payload = {
    'selected_model': metrics.get('selected_model'),
    'cv_roc_auc_mean': metrics.get('cv_roc_auc_mean'),
    'cv_roc_auc_std': metrics.get('cv_roc_auc_std'),
    'cv_ap_mean': metrics.get('cv_ap_mean'),
    'cv_ap_std': metrics.get('cv_ap_std'),
}
with open('../models/model_benchmark.json', 'w', encoding='utf-8') as f:
    json.dump(benchmark_payload, f, ensure_ascii=False, indent=2)
print('Saved ../models/model_benchmark.json')